In [4]:

#import all required lib
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

ratings.head()
#merge movies,rating
data = ratings.merge(movies, on="movieId", how="left")
data.head()

# Create User-Movie Matrix
user_movie_matrix = data.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Compute User Similarity Matrix
user_similarity = cosine_similarity(user_movie_matrix_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [ ]:
# predict rating function
def predict_rating_debug(user_id, movie_id):
    print(f"\nPredicting rating for User {user_id} and Movie {movie_id}")
    
    if movie_id not in user_movie_matrix.columns:
        print("Movie not found in dataset")
        return None

    user_sim_scores = user_similarity_df[user_id]
    movie_ratings = user_movie_matrix[movie_id]

    print("\nSimilarities with other users:")
    print(user_sim_scores.sort_values(ascending=False).head(5))

    print("\nRatings for this movie:")
    print(movie_ratings.dropna().head())

    mask = movie_ratings.notna()

    weighted_sum = np.dot(user_sim_scores[mask], movie_ratings[mask])
    similarity_sum = user_sim_scores[mask].sum()

    print("\nWeighted sum calculation:")
    for uid in movie_ratings[mask].index:
        sim = user_sim_scores[uid]
        rating = movie_ratings[uid]
        print(f"User {uid}: similarity={sim:.3f}, rating={rating}")

    print(f"\nTotal weighted sum: {weighted_sum}")
    print(f"Total similarity sum: {similarity_sum}")

    prediction = weighted_sum / similarity_sum
    print(f"\nPredicted rating: {prediction}")

    return prediction
predict_rating_debug(user_id=1, movie_id=1)


Predicting rating for User 1 and Movie 1

Similarities with other users:
userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
Name: 1, dtype: float64

Ratings for this movie:
userId
1     4.0
5     4.0
7     4.5
15    2.5
17    4.5
Name: 1, dtype: float64

Weighted sum calculation:
User 1: similarity=1.000, rating=4.0
User 5: similarity=0.129, rating=4.0
User 7: similarity=0.159, rating=4.5
User 15: similarity=0.161, rating=2.5
User 17: similarity=0.264, rating=4.5
User 18: similarity=0.215, rating=3.5
User 19: similarity=0.325, rating=4.0
User 21: similarity=0.153, rating=3.5
User 27: similarity=0.239, rating=3.0
User 31: similarity=0.164, rating=5.0
User 32: similarity=0.146, rating=3.0
User 33: similarity=0.152, rating=3.0
User 40: similarity=0.095, rating=5.0
User 43: similarity=0.122, rating=5.0
User 44: similarity=0.111, rating=3.0
User 45: similarity=0.328, rating=4.0
User 46: similarity=0.110, rating=5.0
User 50: similarity=0.106, rating=3.0
U

np.float64(3.9110225386215824)

In [13]:
# Recommendation Function
def recommend_movies_debug(user_id, n=5):
    print(f"\nGenerating recommendations for User {user_id}\n")

    user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = user_ratings[user_ratings.isna()].index

    print(f"Total unrated movies: {len(unrated_movies)}\n")

    predictions = []

    for movie_id in unrated_movies[:20]:  # limit for readability
        print(f"Predicting for Movie ID: {movie_id}")
        rating = predict_rating_debug(user_id, movie_id)

        if rating is not None and not np.isnan(rating):
            predictions.append((movie_id, rating))
            print(f"Predicted Rating: {rating}\n")
        else:
            print("Skipped\n")

    print("\nSorting movies by predicted rating...\n")
    predictions.sort(key=lambda x: x[1], reverse=True)

    print("Top recommended movies:\n")

    for movie_id, rating in predictions[:n]:
        title = movies[movies["movieId"] == movie_id]["title"].values[0]
        print(f"{title} → Predicted Rating: {rating}")

    return predictions[:n]


recommend_movies(user_id=1, n=5)

[('Babes in Toyland (1934)', np.float64(5.000000000000001)),
 ('Galaxy of Terror (Quest) (1981)', np.float64(5.000000000000001)),
 ('Alien Contamination (1980)', np.float64(5.000000000000001)),
 ('Saving Face (2004)', np.float64(5.000000000000001)),
 ('Lamerica (1994)', np.float64(5.0))]